In [55]:
import pandas as pd
import pandas as pd
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import KNNBasic





QUESTION 1.1

In [56]:

df = pd.read_csv("Faceplate.csv")

print(df.head(10))

   Transaction  Red  White  Blue  Orange  Green  Yellow
0            1    1      1     0       0      1       0
1            2    0      1     0       1      0       0
2            3    0      1     1       0      0       0
3            4    1      1     0       1      0       0
4            5    1      0     1       0      0       0
5            6    0      1     1       0      0       0
6            7    1      0     1       0      0       0
7            8    1      1     1       0      1       0
8            9    1      1     1       0      0       0
9           10    0      0     0       0      0       1


QUESTION 1.2

In [57]:
# transactions for both red and white
red_white = df[(df["Red"] == 1) & (df["White"] == 1)]

support_count = len(red_white)
total = len(df)

support = support_count / total

print("Red and White count:", support_count)
print("Total transactions:", total)
print("Support:", support)


Red and White count: 4
Total transactions: 10
Support: 0.4


QUESTION 2.1

In [58]:



df = pd.read_csv("Faceplate.csv")

# drop Transaction column
df = df.drop(columns=["Transaction"])

df = df.astype(bool)

frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)

frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False)

print(frequent_itemsets)


    support                        itemsets
1       0.7              frozenset({White})
0       0.6                frozenset({Red})
2       0.6               frozenset({Blue})
5       0.4         frozenset({White, Red})
6       0.4          frozenset({Blue, Red})
8       0.4        frozenset({Blue, White})
3       0.2             frozenset({Orange})
4       0.2              frozenset({Green})
7       0.2         frozenset({Red, Green})
9       0.2      frozenset({White, Orange})
10      0.2       frozenset({White, Green})
11      0.2   frozenset({Blue, White, Red})
12      0.2  frozenset({White, Red, Green})


QUESTION 2.2

In [59]:
rules = association_rules(frequent_itemsets,
                          metric="confidence",
                          min_threshold=0.5)

rules = rules.sort_values(by="lift", ascending=False)

print(rules)


                  antecedents              consequents  antecedent support  \
12    frozenset({White, Red})       frozenset({Green})                 0.4   
15         frozenset({Green})  frozenset({White, Red})                 0.2   
6          frozenset({Green})         frozenset({Red})                 0.2   
14  frozenset({White, Green})         frozenset({Red})                 0.2   
7         frozenset({Orange})       frozenset({White})                 0.2   
8          frozenset({Green})       frozenset({White})                 0.2   
13    frozenset({Red, Green})       frozenset({White})                 0.2   
2            frozenset({Red})        frozenset({Blue})                 0.6   
3           frozenset({Blue})         frozenset({Red})                 0.6   
0            frozenset({Red})       frozenset({White})                 0.6   
1          frozenset({White})         frozenset({Red})                 0.7   
4          frozenset({White})        frozenset({Blue})          

QUESTION 2.3

In [60]:
top6 = rules.head(6)[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
    "leverage"
]]

print(top6)


                  antecedents              consequents  support  confidence  \
12    frozenset({White, Red})       frozenset({Green})      0.2         0.5   
15         frozenset({Green})  frozenset({White, Red})      0.2         1.0   
6          frozenset({Green})         frozenset({Red})      0.2         1.0   
14  frozenset({White, Green})         frozenset({Red})      0.2         1.0   
7         frozenset({Orange})       frozenset({White})      0.2         1.0   
8          frozenset({Green})       frozenset({White})      0.2         1.0   

        lift  leverage  
12  2.500000      0.12  
15  2.500000      0.12  
6   1.666667      0.08  
14  1.666667      0.08  
7   1.428571      0.06  
8   1.428571      0.06  


QUESTION 2.4

In [61]:
best = rules.iloc[0]

ant = list(best["antecedents"])
con = list(best["consequents"])
conf = best["confidence"] * 100
lift = best["lift"]

sentence = f"If {ant} are purchased, then with confidence {conf:.2f}% {con} will also be purchased. This rule has a lift ratio of {lift:.2f}."

print("\nBEST RULE INTERPRETATION:")
print(sentence)



BEST RULE INTERPRETATION:
If ['White', 'Red'] are purchased, then with confidence 50.00% ['Green'] will also be purchased. This rule has a lift ratio of 2.50.


QUESTION 3.1

In [62]:


df = pd.read_csv("CharlesBookClub.csv")

# ghost non-book columns
ignore_cols = ['Seq#', 'ID#', 'Gender', 'M', 'R', 'F', 'FirstPurch', 'Related Purchase']
book_cols = [col for col in df.columns if col not in ignore_cols]

binary_matrix = (df[book_cols] > 0).astype(int)

print(binary_matrix.head(10))



   ChildBks  YouthBks  CookBks  DoItYBks  RefBks  ArtBks  GeogBks  ItalCook  \
0         0         1        1         0       0       0        0         0   
1         0         0        0         0       0       0        0         0   
2         1         1        1         0       1       0        1         1   
3         0         0        0         0       0       0        0         0   
4         0         0        0         0       0       0        0         0   
5         0         0        0         0       0       0        0         0   
6         0         0        0         0       0       0        1         0   
7         1         0        0         0       0       0        0         0   
8         0         0        0         0       0       0        0         0   
9         0         0        1         0       0       0        0         0   

   ItalAtlas  ItalArt  Florence  Mcode  Rcode  Fcode  Yes_Florence  \
0          0        0         0      1      1      1        

QUESTION 3.2

In [63]:


frequent_itemsets = apriori(binary_matrix, min_support=200/2000, use_colnames=True)

print(f"Number of frequent itemsets : {len(frequent_itemsets)}")


Number of frequent itemsets : 367


/home/ally/anaconda3/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


QUESTION 3.3

In [64]:

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

rules_sorted = rules.sort_values(by='lift', ascending=False)

# show 25 rules
top_25_rules = rules_sorted.head(25)

# Show columns which are needed
top_25_rules_display = top_25_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']]
print(top_25_rules_display)


                                   antecedents  \
1374                       frozenset({RefBks})   
1385                frozenset({Mcode, RefBks})   
468                        frozenset({RefBks})   
467                 frozenset({Mcode, RefBks})   
474                 frozenset({Rcode, RefBks})   
1404                       frozenset({RefBks})   
1403                frozenset({Fcode, RefBks})   
1400                frozenset({Rcode, RefBks})   
2367                       frozenset({RefBks})   
1396         frozenset({Rcode, RefBks, Fcode})   
475                        frozenset({RefBks})   
75                         frozenset({RefBks})   
2366                frozenset({Fcode, RefBks})   
1388                frozenset({Fcode, RefBks})   
2363                frozenset({Mcode, RefBks})   
1389                       frozenset({RefBks})   
2362                frozenset({Rcode, RefBks})   
2355         frozenset({Rcode, RefBks, Fcode})   
482                        frozenset({RefBks})   


/home/ally/anaconda3/lib/python3.11/site-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


QUESTION 4.1

In [65]:


rule_highest_support = rules.loc[rules['support'].idxmax()]

print("4.1 Rule with Highest Support:")
print("Antecedents:", rule_highest_support['antecedents'])
print("Consequents:", rule_highest_support['consequents'])
print("Support:", rule_highest_support['support'])
print("Confidence:", rule_highest_support['confidence'])
print("Lift:", rule_highest_support['lift'])


4.1 Rule with Highest Support:
Antecedents: frozenset({'Rcode'})
Consequents: frozenset({'Mcode'})
Support: 1.0
Confidence: 1.0
Lift: 1.0


QUESTION 4.2

In [66]:

rule_highest_lift = rules.loc[rules['lift'].idxmax()]

print("4.2 Rule with Highest Lift:")
print("Antecedents:", rule_highest_lift['antecedents'])
print("Consequents:", rule_highest_lift['consequents'])
print("Support:", rule_highest_lift['support'])
print("Confidence:", rule_highest_lift['confidence'])
print("Lift:", rule_highest_lift['lift'])

# Compare support values with highest support rule
rule_highest_support = rules.loc[rules['support'].idxmax()]
print("\nComparison of Support Values:")
print("Support of Highest Support Rule:", rule_highest_support['support'])
print("Support of Highest Lift Rule:", rule_highest_lift['support'])



4.2 Rule with Highest Lift:
Antecedents: frozenset({'RefBks'})
Consequents: frozenset({'CookBks', 'ChildBks'})
Support: 0.1035
Confidence: 0.5054945054945055
Lift: 2.0888202706384527

Comparison of Support Values:
Support of Highest Support Rule: 1.0
Support of Highest Lift Rule: 0.1035


QUESTION 4.3

In [67]:

top10_lift = rules.sort_values(by='lift', ascending=False).head(10)

print("Top 10 Rules by Lift:")
print(top10_lift[['antecedents','consequents','support','confidence','lift']])

# Find the rule with lowest confidence among top 10 lift rules
rule_lowest_conf_top10 = top10_lift.loc[top10_lift['confidence'].idxmin()]

print("\nRule with Lowest Confidence among Top 10 Lift Rules:")
print("Antecedents:", rule_lowest_conf_top10['antecedents'])
print("Consequents:", rule_lowest_conf_top10['consequents'])
print("Support:", rule_lowest_conf_top10['support'])
print("Confidence:", rule_lowest_conf_top10['confidence'])
print("Lift:", rule_lowest_conf_top10['lift'])


Top 10 Rules by Lift:
                            antecedents  \
1374                frozenset({RefBks})   
1385         frozenset({Mcode, RefBks})   
468                 frozenset({RefBks})   
467          frozenset({Mcode, RefBks})   
474          frozenset({Rcode, RefBks})   
1404                frozenset({RefBks})   
1403         frozenset({Fcode, RefBks})   
1400         frozenset({Rcode, RefBks})   
2367                frozenset({RefBks})   
1396  frozenset({Rcode, RefBks, Fcode})   

                                            consequents  support  confidence  \
1374       frozenset({Rcode, ChildBks, Mcode, CookBks})   0.1035    0.505495   
1385              frozenset({CookBks, ChildBks, Fcode})   0.1035    0.505495   
468               frozenset({CookBks, ChildBks, Mcode})   0.1035    0.505495   
467                      frozenset({CookBks, ChildBks})   0.1035    0.505495   
474                      frozenset({CookBks, ChildBks})   0.1035    0.505495   
1404       frozenset({Rc

QUESTION 5.1

In [68]:

np.random.seed(0)

num_transactions = 50
num_items = 9

#
data = np.random.randint(0, 2, size=(num_transactions, num_items))

# Create binary incidence matrix
columns = [f'Item{i+1}' for i in range(num_items)]
synthetic_df = pd.DataFrame(data, columns=columns)

print("5.1 Synthetic Binary Incidence Matrix (first 10 rows):")
print(synthetic_df.head(10))


5.1 Synthetic Binary Incidence Matrix (first 10 rows):
   Item1  Item2  Item3  Item4  Item5  Item6  Item7  Item8  Item9
0      0      1      1      0      1      1      1      1      1
1      1      1      0      0      1      0      0      0      0
2      0      1      0      1      1      0      0      1      1
3      1      1      0      1      0      1      0      1      1
4      0      1      1      0      0      1      0      1      1
5      1      1      1      0      1      0      1      1      1
6      1      0      1      0      0      1      1      0      1
7      0      1      0      0      0      0      0      1      1
8      0      0      0      1      1      0      1      0      0
9      1      0      1      1      1      1      1      1      0


QUESTION 5.2

In [69]:



frequent_itemsets = apriori(synthetic_df, min_support=2/50, use_colnames=True)

print("5.2 Frequent Itemsets:")
print(frequent_itemsets)

# Generate association rules with minimum confidence 0.7
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

print("\n5.2 Association Rules (confidence >= 0.7):")
print(rules[['antecedents','consequents','support','confidence','lift']])


/home/ally/anaconda3/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


5.2 Frequent Itemsets:
     support                                           itemsets
0       0.54                                 frozenset({Item1})
1       0.68                                 frozenset({Item2})
2       0.52                                 frozenset({Item3})
3       0.54                                 frozenset({Item4})
4       0.56                                 frozenset({Item5})
..       ...                                                ...
353     0.04  frozenset({Item5, Item7, Item3, Item8, Item4, ...
354     0.04  frozenset({Item5, Item7, Item9, Item3, Item8, ...
355     0.04  frozenset({Item2, Item7, Item1, Item3, Item8, ...
356     0.04  frozenset({Item5, Item7, Item1, Item3, Item8, ...
357     0.04  frozenset({Item2, Item5, Item7, Item9, Item3, ...

[358 rows x 2 columns]

5.2 Association Rules (confidence >= 0.7):
                                        antecedents  \
0                                frozenset({Item8})   
1                              

QUESTION 5.3

In [70]:

top6_rules = rules.sort_values(by='lift', ascending=False).head(6)

print("5.3 Top 6 Rules by Lift (Uplift Opportunities):")
print(top6_rules[['antecedents','consequents','support','confidence','lift']])

high_lift_threshold = 3  
high_lift_rules = top6_rules[top6_rules['lift'] > high_lift_threshold]

print("\nRules with exceptionally high lift (>3):")
print(high_lift_rules[['antecedents','consequents','support','confidence','lift']])


5.3 Top 6 Rules by Lift (Uplift Opportunities):
                                        antecedents  \
376         frozenset({Item7, Item8, Item6, Item9})   
359  frozenset({Item5, Item7, Item3, Item4, Item6})   
368  frozenset({Item2, Item5, Item9, Item3, Item6})   
327         frozenset({Item2, Item4, Item5, Item7})   
285         frozenset({Item8, Item1, Item6, Item3})   
360  frozenset({Item5, Item1, Item3, Item8, Item6})   

                          consequents  support  confidence      lift  
376  frozenset({Item2, Item5, Item3})     0.04         1.0  5.555556  
359         frozenset({Item8, Item1})     0.04         1.0  5.000000  
368         frozenset({Item7, Item8})     0.04         1.0  5.000000  
327         frozenset({Item8, Item6})     0.04         1.0  4.545455  
285         frozenset({Item7, Item4})     0.06         1.0  4.545455  
360         frozenset({Item7, Item4})     0.04         1.0  4.545455  

Rules with exceptionally high lift (>3):
                           

QUESTION 6.1

In [71]:


np.random.seed(0)

n_ratings = 5000

# Generate synthetic data
user_ids = np.random.randint(0, 1000, n_ratings)
item_ids = np.random.randint(0, 100, n_ratings)
ratings = np.random.randint(1, 6, n_ratings)

ratings_df = pd.DataFrame({
    'userID': user_ids,
    'itemID': item_ids,
    'rating': ratings
})

print("6.1 First 10 rows of synthetic dataset:")
print(ratings_df.head(10))


6.1 First 10 rows of synthetic dataset:
   userID  itemID  rating
0     684      49       2
1     559      63       3
2     629       9       5
3     192      24       1
4     835      68       4
5     763      26       5
6     707      52       1
7     359      54       1
8       9      85       4
9     723      78       5


QUESTION 6.2

In [72]:


reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(ratings_df[['userID','itemID','rating']], reader)

# Split data
trainset, testset = train_test_split(data, test_size=0.2, random_state=0)

print("6.2 Training set size:", trainset.n_ratings)
print("6.2 Test set size:", len(testset))


6.2 Training set size: 4000
6.2 Test set size: 1000


QUESTION 6.3

In [73]:


# User cosine similarity (for analysis)
user_sim_options = {
    'name': 'cosine',
    'user_based': True
}

user_model = KNNBasic(sim_options=user_sim_options)
user_model.fit(trainset)

print("6.3 User cosine similarity model built.")

item_sim_options = {
    'name': 'cosine',
    'user_based': False
}

item_model = KNNBasic(sim_options=item_sim_options)
item_model.fit(trainset)

print("6.3 Item-based collaborative filtering model built.")


Computing the cosine similarity matrix...
Done computing similarity matrix.
6.3 User cosine similarity model built.
Computing the cosine similarity matrix...
Done computing similarity matrix.
6.3 Item-based collaborative filtering model built.


QUESTION 6.4

In [74]:
# Get all users and items
all_users = ratings_df['userID'].unique()
all_items = ratings_df['itemID'].unique()

# Existing user-item pairs
existing_pairs = set(zip(ratings_df.userID, ratings_df.itemID))

predictions = []

# forecast only missing pairs
for u in all_users:
    for i in all_items:
        if (u,i) not in existing_pairs:
            pred = item_model.predict(u, i)
            predictions.append(pred)

# change predictions to DataFrame
pred_df = pd.DataFrame({
    'userID':[p.uid for p in predictions],
    'itemID':[p.iid for p in predictions],
    'pred_rating':[p.est for p in predictions]
})

# Top recommendation per every user
recommendations = pred_df.sort_values(['userID','pred_rating'], ascending=[True,False])\
                          .groupby('userID').head(1)

print("6.4 Recommended item for each user:")
print(recommendations.head(20))


6.4 Recommended item for each user:
       userID  itemID  pred_rating
46304       0      48     3.733805
90607       1      68     2.982750
81450       2      39     3.918365
43967       3      31     5.000000
35774       4      24     4.664973
85132       5      16     4.654970
62442       6       0     2.982750
41039       7      44     5.000000
76840       8      29     3.000000
784         9      41     4.000000
74176      10      50     5.000000
15904      11      61     3.698071
59672      12       0     5.000000
33850      13       5     3.344168
92965      14      58     3.000000
73252      15      16     4.000000
32554      16      97     4.215903
55641      17      53     3.500000
89829      18      49     2.982750
18006      19      73     5.000000
